# The Diagnostics Playbook

*Companion to* **Mastering CatBoost — Pro Edition**, Chapter 4: Tuning & Diagnostics as System Control, §4.5 (Diagnostics: Reading the Heartbeat), §4.6 (Interpretability: The Ladder of Truth), and §4.8 (The Cookbook).

This is the systematic model-debugging workflow of Chapter 4, packaged as a repeatable procedure: **data audit → baseline → leakage checks → error slicing → decision**. To make every check fire visibly, we build a deliberately booby-trapped dataset containing the two most common production pathologies — a leaky proxy feature and an ID-style high-cardinality column — plus class imbalance, missing values, and a weak segment. Your job in real projects is to run this same sequence on data whose traps you *don't* know in advance.

In [ ]:
%pip install -q catboost==1.2.10 scikit-learn==1.6.1 numpy==2.0.2 pandas==2.2.2 matplotlib==3.10.0

import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from catboost import CatBoostClassifier, Pool

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

## 0. The booby-trapped dataset

Ground truth (known to us, hidden from the workflow):

- 8 legitimate numeric features carry the real signal; the positive rate is ~15% (imbalance, Cookbook Scenario 3).
- `leaky_score` is a post-outcome proxy: essentially the target plus noise (think "payment status" leaking into a default model).
- `customer_id` has ~1 row per level — pure memorization fuel (Cookbook Scenario 1).
- `segment` ∈ {A, B, C}: in segment C the signal is halved and noisier (the weak slice for error slicing).
- ~8% missing values in two numeric columns.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss
from sklearn.model_selection import train_test_split

n = 15_000
rng = np.random.RandomState(RANDOM_SEED)

X_num, y = make_classification(n_samples=n, n_features=8, n_informative=6, n_redundant=1,
                               weights=[0.85], flip_y=0.06, random_state=RANDOM_SEED)
df = pd.DataFrame(X_num, columns=[f"f{i}" for i in range(8)])

# Segment C has extra label noise; this changes the attainable signal.
segment = rng.choice(list("ABC"), size=n, p=[0.4, 0.4, 0.2])
seg_c = segment == "C"
flip = seg_c & (rng.rand(n) < 0.15)
y = np.where(flip, 1 - y, y)

df["segment"] = segment
df["leaky_score"] = y * 0.8 + rng.normal(0, 0.35, n)          # post-outcome proxy
df["customer_id"] = (rng.permutation(n) // 1.2).astype(int).astype(str)  # ~1 row/level
df.loc[rng.rand(n) < 0.08, "f2"] = np.nan
df.loc[rng.rand(n) < 0.08, "f5"] = np.nan
df["y"] = y

train_df, valid_df = train_test_split(df, test_size=0.3, random_state=RANDOM_SEED, stratify=df["y"])
train_df, valid_df = train_df.reset_index(drop=True), valid_df.reset_index(drop=True)
FEATURES = [c for c in df.columns if c != "y"]
CATS = ["segment", "customer_id"]
print(train_df.shape, valid_df.shape, "| positive rate:", round(df.y.mean(), 3))

## Step 1 — Data audit (Level 0 of the Hierarchy, §4.2.1)

Before any model: types, missingness, cardinality, target balance, duplicates. The audit's job is to surface *contract violations* — the problems no hyperparameter can fix. Two red flags to know by heart:

1. A categorical with cardinality ≈ row count is an ID, not a feature.
2. Any feature whose meaning is only known *after* the outcome (audit question: "would this value exist at prediction time?").

In [ ]:
audit = pd.DataFrame({
    "dtype": train_df[FEATURES].dtypes.astype(str),
    "n_missing": train_df[FEATURES].isna().sum(),
    "pct_missing": (train_df[FEATURES].isna().mean() * 100).round(1),
    "n_unique": train_df[FEATURES].nunique(),
})
audit["cardinality_ratio"] = (audit["n_unique"] / len(train_df)).round(3)
print(audit)
print(f"\ntarget balance : {train_df.y.mean():.3f} positive (imbalanced -> Scenario 3 applies)")
print(f"duplicated rows: {train_df.duplicated().sum()}")
print(f"train/valid target rates: {train_df.y.mean():.3f} / {valid_df.y.mean():.3f}")

flagged_ids = audit.index[(audit["cardinality_ratio"] > 0.5) & (audit["dtype"] == "object")].tolist()
print(f"\nAUDIT FLAG — ID-like categorical(s): {flagged_ids}")
print("AUDIT QUESTION — for every feature: is it available BEFORE the outcome? "
      "('leaky_score' needs a domain answer; the model will give us a statistical one next.)")

## Step 2 — Baseline

Start with native categorical handling, a generous iteration ceiling, and validation-based early stopping. Class imbalance alone does not justify automatic class weighting: weighting changes the training objective and probability scale. Begin unweighted, then compare weighting only against the deployment metric, calibration, threshold, and decision cost. The all-feature baseline remains a diagnostic instrument, not a production candidate.

In [ ]:
def make_pools(feats):
    cats = [c for c in CATS if c in feats]
    return (Pool(train_df[feats], train_df["y"], cat_features=cats),
            Pool(valid_df[feats], valid_df["y"], cat_features=cats))

def fit_report(feats, name, **extra):
    tr_pool, va_pool = make_pools(feats)
    params = dict(iterations=800, learning_rate=0.05, depth=6, l2_leaf_reg=3,
                  eval_metric="AUC",
                  random_seed=RANDOM_SEED, verbose=0, allow_writing_files=False)
    params.update(extra)
    m = CatBoostClassifier(**params)
    m.fit(tr_pool, eval_set=va_pool, early_stopping_rounds=60, use_best_model=True)
    p_tr = m.predict_proba(train_df[feats])[:, 1]
    p_va = m.predict_proba(valid_df[feats])[:, 1]
    rep = {"model": name,
           "train_AUC": round(roc_auc_score(train_df.y, p_tr), 4),
           "valid_AUC": round(roc_auc_score(valid_df.y, p_va), 4),
           "valid_PR_AUC": round(average_precision_score(valid_df.y, p_va), 4),
           "trees": m.tree_count_}
    rep["gap"] = round(rep["train_AUC"] - rep["valid_AUC"], 4)
    return m, rep

reports = []
m_base, rep = fit_report(FEATURES, "baseline (all features)")
reports.append(rep)
pd.DataFrame(reports)

A suspiciously high validation AUC is itself a symptom — Scenario 1's signature is "training score 0.99" but a *leak that survives the split* (like our proxy) inflates validation too. Which is why the next step interrogates the model rather than trusting the score.

## Step 3 — Leakage checks (§4.5.2 + Ladder of Truth, §4.6)

Three probes, in increasing cost:

1. **Importance dominance** (Rung 1, `PredictionValuesChange`): the book's CTR-domination test — any feature above ~50% share is a suspect.
2. **Curve shape**: a validation curve that jumps almost immediately to its ceiling means the model found a shortcut, not structure.
3. **Ablation** (Rung 2 logic): remove the suspect, retrain, and watch the metric. A *large legitimate* feature survives ablation with a modest drop; a *leak* takes its miracle with it.

In [ ]:
imp = m_base.get_feature_importance(type="PredictionValuesChange", prettified=True)
imp["share_%"] = (imp["Importances"] / imp["Importances"].sum() * 100).round(1)
print(imp.to_string(index=False))

dominant = imp.loc[imp["share_%"] > 50, "Feature Id"].tolist()
print(f"\nREVIEW CUE — importance share > 50%: "
      f"{dominant if dominant else 'none'}")

curve = m_base.get_evals_result()["validation"]["AUC"]
frac_at_10 = curve[min(9, len(curve)-1)] / max(curve)
print(f"validation AUC after 10 trees = {100*frac_at_10:.1f}% of its maximum "
      "(an early concentration is a review cue, not proof of leakage)")

In [ ]:
# Probe 3: ablation of the top suspect
suspects = dominant if dominant else [imp.iloc[0]["Feature Id"]]
feats_clean = [f for f in FEATURES if f not in suspects]
m_clean, rep = fit_report(feats_clean, f"without {suspects}")
reports.append(rep)
pd.DataFrame(reports)

Ablation measures dependence of validation performance on a feature; it cannot determine whether the feature is legitimate or available at decision time. Here domain provenance says `leaky_score` is post-outcome, so it must be removed regardless of importance share or ablation size.

### The ID column

`customer_id` is nearly unique and has no declared serving-time generalization role in this fixture. Dropping it is the clean control. `max_ctr_complexity=1` changes interaction construction but does not make an identifier legitimate or prevent all memorization. Keep an identifier only if the deployment target includes recurring entities and validation on the intended future entities demonstrates legitimate value.

In [ ]:
feats_no_leak = [f for f in FEATURES if f != "leaky_score"]
feats_no_id = [f for f in feats_no_leak if f != "customer_id"]

_, rep = fit_report(feats_no_leak, "no leak, with customer_id")
reports.append(rep)
_, rep = fit_report(feats_no_leak, "no leak, id throttled (max_ctr_complexity=1)",
                    max_ctr_complexity=1)
reports.append(rep)
m_final, rep = fit_report(feats_no_id, "no leak, id dropped")
reports.append(rep)
pd.DataFrame(reports)

Compare the ID-kept, throttled, and dropped rows using validation utility and the intended deployment target. In this generator the ID has no signal, but cardinality alone does not prove an identifier is useless for every recurring-entity problem. Throttling is a model-capacity experiment, not a leakage remedy.

## Step 4 — Error slicing

Slice validation performance by relevant groups and prediction ranges. Treat small-slice estimates with uncertainty.

In [ ]:
p_va = m_final.predict_proba(valid_df[feats_no_id])[:, 1]
slices = []
for seg, grp_idx in valid_df.groupby("segment").groups.items():
    yy = valid_df.loc[grp_idx, "y"]
    pp = pd.Series(p_va, index=valid_df.index).loc[grp_idx]
    slices.append({"segment": seg, "n": len(yy),
                   "pos_rate": round(yy.mean(), 3),
                   "AUC": round(roc_auc_score(yy, pp), 4),
                   "logloss": round(log_loss(yy, pp, labels=[0, 1]), 4)})
slice_table = pd.DataFrame(slices)
print(slice_table.to_string(index=False))

worst = slice_table.loc[slice_table["AUC"].idxmin(), "segment"]
print(f"\nSLICING FLAG — weakest segment: {worst} "
      "(smaller AUC and higher logloss than siblings)")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].bar(slice_table["segment"], slice_table["AUC"], color="steelblue")
axes[0].set_title("validation AUC by segment"); axes[0].set_ylim(0.5, 1.0)

deciles = pd.qcut(p_va, 10, labels=False, duplicates="drop")
dec = pd.DataFrame({"decile": deciles, "pred": p_va, "actual": valid_df["y"].to_numpy()}).groupby("decile").mean()
axes[1].plot(dec["pred"], dec["actual"], "o-")
axes[1].plot([0, 1], [0, 1], "--", color="gray")
axes[1].set_xlabel("mean predicted probability"); axes[1].set_ylabel("observed rate")
axes[1].set_title("decile reliability (diagnostic only)")
plt.tight_layout(); plt.show()

Two independent findings to log:

1. Inspect the live slice table and its sample sizes. Segment C was generated with extra label flips, but a real investigation would not know the cause in advance; candidate actions require provenance, uncertainty, and validation.
2. `m_final` is unweighted. Its reliability curve is therefore descriptive evidence from this split and cannot be attributed to `auto_class_weights`. If calibrated probabilities matter, assess a proper score and reliability on dedicated data; calibrate only after the candidate model is frozen.

## Step 5 - Decision

The playbook ends with an explicit, recorded decision.

In [ ]:
ledger = pd.DataFrame(reports)
ledger

**Decision template (populate from evidence and provenance):**

| Item | Evidence | Governed action |
|---|---|---|
| post-outcome field | availability/provenance plus ablation | remove before modeling |
| near-unique ID | deployment estimand and group-aware validation | drop by default; retain only with a legitimate recurring-entity contract |
| imbalance | prevalence, calibration, costs, threshold metrics | compare unweighted/weighted objectives; no automatic weighting rule |
| weak segment | sample size and uncertainty | investigate, collect data, or monitor |

## Takeaways

- Leakage is determined by feature timing and provenance, not an arbitrary importance threshold.
- Importance dominance and early saturation are review cues; ablation measures dependence, not legitimacy.
- Dropping a pure identifier is the control. Lowering `max_ctr_complexity` does not sanitize it.
- Weighting must be justified by validation decision metrics and recalibration needs.
- Keep a written decision ledger with evidence, uncertainty, and owner.

## Exercises

**Exercise 1.** Add a weaker post-outcome proxy and compare review cues. Explain why missing a 50% threshold does not clear the feature.

<details><summary>Solution</summary>

Run importance, learning curves, and ablation, then use provenance to decide. Thresholds prioritize review; they never establish temporal availability.
</details>

**Exercise 2.** Compare PredictionValuesChange with LossFunctionChange on a validation Pool. Report disagreement and uncertainty; neither importance type supplies causal or leakage proof.

<details><summary>Solution</summary>

```python
_, va_pool = make_pools(feats_no_id)
lfc = m_final.get_feature_importance(va_pool, type="LossFunctionChange", prettified=True)
pvc = m_final.get_feature_importance(type="PredictionValuesChange", prettified=True)
print(pd.merge(pvc, lfc, on="Feature Id", suffixes=("_PVC", "_LFC")))
```
</details>

**Exercise 3.** Inspect interactions for the ID-kept model. Treat high scores as descriptions of fitted use, then decide from deployment legitimacy and group-aware validation.

<details><summary>Solution</summary>

`max_ctr_complexity=1` can suppress some combination CTRs, but it does not prove an ID is safe. Compare the ID-dropped control.
</details>